# 03 Перенос ACT на ROCm и диагностика DAgger

        Этот раздел на примере ACT показывает: снижение loss при обучении не означает успеха в closed-loop rollout. Сначала смотрим кривые диагностики, затем связываем воспроизведение данных, open-loop, closed-loop, видео с провалами и DAgger в одну цепочку поиска неисправностей.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: кривые диагностики ACT


In [ ]:
snapshot = json.loads((ASSET_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
rows = []
for item in snapshot["act_progress"]:
    rows.append((item["label"], f'{item["success"]}/{item["total"]}', f'{item["rate"]:.1%}'))
md_table(["Этап", "physical_success", "Успешность"], rows)
show_asset("act_dagger_progress_curve.png", width=960)


Эта кривая нужна не для того, чтобы доказать идеальную генерализацию ACT, а чтобы показать направление поиска неисправностей: сначала убедиться в данных и состоянии closed-loop, и только потом решать, нужен ли DAgger.


## Checkpoint 2: шаблон команды строгой оценки ACT


In [ ]:
act_policy = MODEL_ROOT / "act_best" / "step_5000"
cmd = f"""
cd "$PROJECT_ROOT"
export PYTHONPATH="$PWD:${{PYTHONPATH:-}}"
./.venv/bin/python eval_policy_success.py \\
  --policy act \\
  --act-policy-path "{act_policy}" \\
  --physical-success \\
  --seed-start 1000 \\
  --episodes 10 \\
  --max-action-steps 1200 \\
  --output-jsonl "$OUTPUT_ROOT/act_seen_1000_1009.jsonl"
"""
print(cmd)


Полное обучение и пакетную оценку лучше запускать в терминале или фоновом скрипте. Здесь Notebook сохраняет шаблон команды, объясняет смысл параметров и читает результаты.


## Checkpoint 3: сравнение видео успеха и провала


In [ ]:
show_asset("act_success_sequence.jpg", width=1100)
show_asset("act_failure_sequence.jpg", width=1100)


Провалы ACT часто случаются не из-за полной неподвижности, а из-за сбоя на одном из этапов: приближении, захвате, переносе или освобождении. При проверке видео нужно записать, на каком этапе произошёл провал.


## Checkpoint 4: запись эксперимента DAgger


In [ ]:
rows = [
    ("Длина prefix", "например 40 control tick"),
    ("Тип oracle", "scripted policy / human correction"),
    ("timestamp offset", "например, для correction episode используйте 2.0"),
    ("Вес выборки", "например, correction episode weight=0.25"),
    ("Seed оценки", "seen / mixed / heldout указывайте отдельно"),
    ("Условие замены базовой модели", "должна превзойти старый baseline по тому же physical_success"),
]
md_table(["Пункт записи", "Пример"], rows)
